## Advanced OOP in Python: Class Methods, Static Methods, and Constructor `Overloading`

**What you will learn:**
- What **class methods** are, and the factory-method pattern they enable
- What **static methods** are, and how they differ from class methods
- Why Python **doesn't support true constructor overloading**, and the three
  standard ways Python programmers work around that

## 1. Class Methods

- A class method is bound to the **class itself**, not to any one instance.
- It receives the class as its first parameter, conventionally called **`cls`**
  (the same role `self` plays for instance methods).
- It can read and modify **class-level variables** - state shared by every
  instance - but it has no access to any particular instance's own data.
- You create one with the **`@classmethod`** decorator.

**When to use it:** whenever a method's job is about the *class as a whole*
(e.g. changing a setting shared by everyone, or building a new instance in a
particular way), rather than about one specific object.


In [1]:
class Student:
    school_name = "ABC School"   # a CLASS variable -- shared by every instance

    def __init__(self, name, age):
        self.name = name          # these are INSTANCE variables -- unique per object
        self.age = age

    @classmethod
    def change_school(cls, new_name):
        cls.school_name = new_name   # changes it for EVERY Student, present and future


jessa = Student("Jessa", 14)
print("Before:", jessa.school_name)

Student.change_school("XYZ School")   # called on the class, not an instance

tommy = Student("Thomson", 15)        # created AFTER the change
print("After: ", jessa.school_name)   # jessa sees the change too
print("After: ", tommy.school_name)


Before: ABC School
After:  XYZ School
After:  XYZ School


**What just happened:** `school_name` belongs to the class, not to `jessa`
or `tommy` individually. Calling `change_school()` updates it once, and
**every** instance - old and new - sees the new value.


### The factory-method pattern

A very common, genuinely useful use of `@classmethod` is building an
**alternate constructor** - a second, clearly-named way to create an object,
when the normal `__init__` signature isn't a natural fit for some inputs.


In [2]:
from datetime import date

class Student:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    @classmethod
    def from_birth_year(cls, name, birth_year):
        # Work out the age, then build and return a normal Student object.
        # cls(...) calls the constructor -- exactly like Student(...) would.
        age = date.today().year - birth_year
        return cls(name, age)

    def show(self):
        print(f"{self.name}'s age is {self.age}")


jessa = Student("Jessa", 20)               # the usual way
jessa.show()

joy = Student.from_birth_year("Joy", 1999)  # the alternate way -- reads cleanly!
joy.show()


Jessa's age is 20
Joy's age is 27


**Why this is useful:** `Student.from_birth_year("Joy", 1999)` is far more
readable at the call site than forcing every caller to calculate the age
themselves before constructing a `Student`. The class method does that
calculation once, in one place, and still ends up calling the normal
constructor (`cls(name, age)`) to build the object.

**We'll come back to this exact pattern in Section 3 — it's the cleanest
solution to Python's lack of constructor overloading.**


### Class methods and inheritance

A useful, slightly subtle behaviour: when a **subclass** calls an inherited
class method, `cls` refers to the **subclass**, not the original parent —
so the factory method still builds the *correct* type of object.


In [4]:
class Vehicle:
    def __init__(self, name, price):
        self.name = name
        self.price = price

    @classmethod
    def from_usd(cls, name, price_usd):
        # cls(...) will build whichever class this was actually called on
        return cls(name, price_usd * 1.27)   # a rough USD-to-GBP conversion

    def show(self):
        print(f"{self.name}: £{self.price:,.2f}")


class Car(Vehicle):
    def show_with_type(self):
        print(f"{self.name} (a car): £{self.price:,.2f}")


# Called on Car, not Vehicle -- so cls is Car, and we get a Car instance back
my_car = Car.from_usd("Model X", 50000)
print(type(my_car))
my_car.show_with_type()


<class '__main__.Car'>
Model X (a car): £63,500.00


## 2. Static Methods

- A static method takes **neither `self` nor `cls`** — it has no automatic
  access to the instance or the class.
- It's really just a regular, standalone function that happens to live
  *inside* a class, grouped there because it's logically related.
- It **cannot** read or modify instance or class state directly.
- You create one with the **`@staticmethod`** decorator.

**When to use it:** for a self-contained utility/helper task that's
conceptually related to the class, but doesn't need to know about any
specific object or any class-level state.


In [3]:
class Employee:
    def __init__(self, name, project_name):
        self.name = name
        self.project_name = project_name

    @staticmethod
    def gather_requirements(project_name):
        # This logic doesn't need self OR cls -- it's a pure lookup based
        # only on the argument passed in. A perfect fit for @staticmethod.
        if project_name == "ABC Project":
            return ["design", "build", "test"]
        return ["general task"]

    def work(self):
        # an instance method CAN still call a static method, via self.
        tasks = self.gather_requirements(self.project_name)
        for task in tasks:
            print(f"{self.name} is doing: {task}")


emp = Employee("Kelly", "ABC Project")
emp.work()




Kelly is doing: design
Kelly is doing: build
Kelly is doing: test


### Class methods vs. static methods, side by side

| | Class method | Static method |
|---|---|---|
| Decorator | `@classmethod` | `@staticmethod` |
| First parameter | `cls` (the class) | *(none - no automatic first argument)* |
| Can read/modify class state? | ✅ Yes | ❌ No |
| Can read/modify instance state? | ❌ No | ❌ No |
| Typical use | Factory methods, changing shared/class-level settings | Self-contained helper logic related to the class |


## 3. Constructor `Overloading` in Python

In languages like Java or C++, you can define **multiple constructors** with
different parameter lists, and the right one is chosen automatically based on
how many/what type of arguments you pass.

**Python does not support this.** A class can only ever have **one**
`__init__` method. Let's see exactly what happens if you try to write more
than one anyway.


In [5]:
class Greeting:
    def __init__(self):
        print("First __init__ defined")

    def __init__(self):
        print("Second __init__ defined")

    # def __init__(self):
    #     print("Third __init__ defined")

    


g = Greeting()


Second __init__ defined


In [10]:
def checking(a,b,c = 10, d = 2):
    return a + b + c + d

checking(1,2)
checking(11,5,6)

24

**What happened:** only `"Third __init__ defined"` printed. In Python, a
later method definition with the same name **completely replaces** the
earlier one — they don't combine or "overload" each other at all. By the
time `Greeting()` runs, the first two `__init__` definitions no longer exist;
only the last one survives.

This is true for **any** method, not just `__init__` — defining the same
method name twice in a class body just means the second definition wins.

So how do Python programmers handle "I want to construct this object in
several different ways"? Three standard approaches, from simplest to most
elegant:


### Solution 1 - Default parameter values + checks inside `__init__`

Give optional parameters a default value of `None`, then branch on what was
actually supplied.


In [7]:
class Booking:
    def __init__(self, name, date, num_guests=1, notes=None):
        self.name = name
        self.date = date
        self.num_guests = num_guests
        self.notes = notes if notes is not None else "No special requests"

    def show(self):
        print(f"{self.name}, {self.date}, guests={self.num_guests} -- {self.notes}")


# All of these are valid -- just with different amounts of detail supplied
Booking("Amara", "2026-07-01").show()
Booking("Femi", "2026-07-02", num_guests=4).show()
Booking("Sam", "2026-07-03", num_guests=2, notes="Window seat please").show()


Amara, 2026-07-01, guests=1 -- No special requests
Femi, 2026-07-02, guests=4 -- No special requests
Sam, 2026-07-03, guests=2 -- Window seat please


**Good for:** a small number of genuinely optional extra details on an
otherwise-consistent set of arguments.

**Limitation:** if the different "versions" of construction take
fundamentally *different kinds* of arguments (not just optional extras), the
`if`/`elif` logic inside `__init__` quickly turns into a tangled mess.


### Solution 2 — `*args` for a flexible number of arguments

In [7]:
class Total:
    def __init__(self, *numbers):
        # *numbers collects ANY number of positional arguments into a tuple
        self.numbers = numbers
        self.total = sum(numbers)

    def show(self):
        print(f"{self.numbers} -> total = {self.total}")


Total(1, 2).show()
Total(3, 4, 5, 6).show()


(1, 2) -> total = 3
(3, 4, 5, 6) -> total = 18


**Good for:** when the *number* of arguments varies, but they're all the
same *kind* of thing (here, numbers to add up).


### Solution 3 - Class method factory functions (the cleanest option)

This is exactly the pattern from Section 1: keep **one** simple `__init__`,
then add clearly-named `@classmethod` "alternate constructors" for each
different way you want to build the object.


In [9]:
class Person:
    def __init__(self, first_name, middle_name, surname):
        self.first_name = first_name
        self.middle_name = middle_name
        self.surname = surname

    @classmethod
    def without_middle_name(cls, first_name, surname):
        return cls(first_name, "", surname)

    @classmethod
    def unknown(cls):
        return cls("", "", "")

    def show(self):
        full = " ".join(part for part in [self.first_name, self.middle_name, self.surname] if part)
        print(full if full else "(unknown person)")


Person("Ada", "Marie", "Lovelace").show()      # the full, normal way
Person.without_middle_name("Alan", "Turing").show()   # a clear alternate way
Person.unknown().show()                         # another clear alternate way


Ada Marie Lovelace
Alan Turing
(unknown person)


**Why this is the best option for genuinely different construction styles:**

- Each alternate constructor has a **descriptive name** — `from_birth_year`,
  `without_middle_name`, `unknown` — instead of relying on guesswork from
  argument types or counts.
- The real `__init__` stays **simple and uncluttered** — no tangled `if`/`elif`
  branching to maintain.
- It reads clearly at the call site: `Person.without_middle_name(...)` tells
  you exactly what you're getting, which plain overloading in other
  languages often doesn't.


---
## Summary

| Concept | Decorator | First parameter | Key use |
|---|---|---|---|
| Class method | `@classmethod` | `cls` | Factory methods; reading/changing shared class state |
| Static method | `@staticmethod` | *(none)* | Self-contained helper logic related to the class |
| Constructor "overloading" | — | — | Python only allows **one** `__init__`; use default args, `*args`, or `@classmethod` factories instead |

### Try these:

- Add a second `@classmethod` factory to the `Vehicle`/`Car` example that
  builds a vehicle `from_eur` instead of `from_usd`.
- Add a `@staticmethod` to `Booking` that validates a date string is in
  `YYYY-MM-DD` format, and call it from inside `__init__`.
- Rewrite `Total` (Solution 2) so it also accepts an optional `label` keyword
  argument, combining `*args` with a regular keyword argument in one
  `__init__` signature.
